In [179]:
import numpy as np
import pandas as pd
import pandera.pandas as pa

import seaborn as sns
import matplotlib.pyplot as plt


In [180]:
df = pd.read_csv("../data/2014-2025data.csv")

/var/folders/57/j7359fzj3n7g0sxmfg939lb40000gn/T/ipykernel_43572/3312239484.py:1: DtypeWarning: Columns (0: Line) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/2014-2025data.csv")


Let's start with some initial profiling of the data.

In [181]:
print('Rows:', len(df), 'Columns:', df.shape[1])
mem = df.memory_usage(deep=True).sum() / (1024**2)
print(f'Memory MB: {mem:.2f}')
display(df.head(5))
display(df.tail(5))
display(df.sample(5))
display(df.describe(include='all').T)

Rows: 161644 Columns: 10
Memory MB: 26.47


,Date,Time,Day,Line,Station,Bound,Vehicle,Incident,Min Delay,Min Gap
0,2014-01-02,06:31:00,Thursday,505.0,Dundas and Roncesvalles,E/B,4018.0,Late Leaving Garage,4.0,8.0
1,2014-01-02,12:43:00,Thursday,504.0,King and Shaw,E/B,4128.0,Utilized Off Route,20.0,22.0
2,2014-01-02,14:01:00,Thursday,501.0,Kingston road and Bingham,W/B,4016.0,Held By,13.0,19.0
3,2014-01-02,14:22:00,Thursday,504.0,King St. and Roncesvalles Ave.,W/B,4175.0,Investigation,7.0,11.0
4,2014-01-02,16:42:00,Thursday,504.0,King and Bathurst,E/B,4080.0,Utilized Off Route,3.0,6.0


,Date,Time,Day,Line,Station,Bound,Vehicle,Incident,Min Delay,Min Gap
161639,2025-12-31,21:49,Wednesday,504 KING,CHERRY AND FRONT,W,4643.0,MTSAN,10.0,20.0
161640,2025-12-31,22:33,Wednesday,505 DUNDAS,PARLIAMENT AND DUNDAS,N,4511.0,MTTO,10.0,20.0
161641,2025-12-31,22:51,Wednesday,505 DUNDAS,BAY AND DUNDAS,N,4453.0,STDP,10.0,20.0
161642,2025-12-31,23:31,Wednesday,501 QUEEN,YORK AND ADELAIDE,E,4547.0,MTSAN,20.0,40.0
161643,2025-12-31,23:52,Wednesday,511 BATHURST,FLEET AND BATHURST,W,4481.0,TTSW,5.0,10.0


,Date,Time,Day,Line,Station,Bound,Vehicle,Incident,Min Delay,Min Gap
107248,2022-04-18,00:20,Monday,511,QUEEN AND BATHURST,W,4515.0,Overhead,26.0,36.0
133446,2024-01-09,21:33,Tuesday,505,KINGSTON LOOP,E,4482.0,Emergency Services,0.0,0.0
156214,2025-08-04,01:46,Monday,505 DUNDAS,DUNDAS AND JARVIS,W,4405.0,MTUS,40.0,70.0
38777,2017-02-05,11:28:00,Sunday,501.0,Roncy and Queen,W/B,4237.0,Investigation,6.0,12.0
155605,2025-07-18,17:28,Friday,505 DUNDAS,BROADVIEW STN TO DUNDA,NaN,4409.0,MTGD,0.0,0.0


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Date,161644,4339,2017-12-28,153,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Time,161638,2880,21:00:00,298,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Day,161644,7,Friday,25182,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Line,160994.0,349.0,501.0,16405.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Station,161380,24285,DUNDAS WEST STATION,2395,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Bound,147792,109,W/B,32579,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Vehicle,156959.0,NaN,NaN,NaN,4601.216687,2201.948495,0.0,4126.0,4434.0,4555.0,163242.0
Incident,161643,136,Mechanical,46545,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Min Delay,161565.0,NaN,NaN,NaN,14.014879,33.253852,0.0,5.0,8.0,11.0,1400.0
Min Gap,161523.0,NaN,NaN,NaN,20.527727,35.516589,0.0,10.0,16.0,20.0,4216.0


The most frequent station is Dundas West Station and the most frequent line appears to be 501 which may indicate that this line experiences the most delays. We will need to do quite a bit of cleaning first. The entries in each column need to be converted into appropriate types and standardized in each column. Let's look at what the entries are right now. 

In [182]:
display(df.info())
for col in df.columns:
    print(df[col].map(type).value_counts())
    print()

<class 'pandas.DataFrame'>
RangeIndex: 161644 entries, 0 to 161643
Data columns (total 10 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   Date       161644 non-null  str    
 1   Time       161638 non-null  str    
 2   Day        161644 non-null  str    
 3   Line       160994 non-null  object 
 4   Station    161380 non-null  str    
 5   Bound      147792 non-null  str    
 6   Vehicle    156959 non-null  float64
 7   Incident   161643 non-null  str    
 8   Min Delay  161565 non-null  float64
 9   Min Gap    161523 non-null  float64
dtypes: float64(3), object(1), str(6)
memory usage: 20.8+ MB


None

Date
<class 'str'>    161644
Name: count, dtype: int64

Time
<class 'str'>      161638
<class 'float'>         6
Name: count, dtype: int64

Day
<class 'str'>    161644
Name: count, dtype: int64

Line
<class 'str'>      95458
<class 'float'>    66186
Name: count, dtype: int64

Station
<class 'str'>      161380
<class 'float'>       264
Name: count, dtype: int64

Bound
<class 'str'>      147792
<class 'float'>     13852
Name: count, dtype: int64

Vehicle
<class 'float'>    161644
Name: count, dtype: int64

Incident
<class 'str'>      161643
<class 'float'>         1
Name: count, dtype: int64

Min Delay
<class 'float'>    161644
Name: count, dtype: int64

Min Gap
<class 'float'>    161644
Name: count, dtype: int64



Let's start with the timing columns: Day, Time, Date. I will check for any invalid entries/null values and then drop those first. Next, store the Time and Date as a single date time object "Datetime" and store the day of the week as a numerical value between 0-7, where Monday = 0, Tuesday = 1, ... , Sunday = 7. 

In [183]:

## We need to check for null values for each category, but it will be helpful to be able to check the count before dropping these values just in case  
## To check the count of null entries in df[col] (without dropping), set drop = False, to drop the null entries, set drop = True.
def null_count(df,col, drop = False):
    s = df[col]
    init_row_count = len(df)
    to_drop = s.isna()
    drop_row_count = to_drop.sum()
    if drop == False:
        print("Total number of entries:", init_row_count)
        print("Number of entries with NA in", col," : ", drop_row_count)
    if drop == True: 
        df = df.loc[~s.isna()]
        print(drop_row_count, " rows have been dropped.")
    return df

## We will also need to check that entries are contained in a certain list of permitted values so let's define a function for that 
## e.g. for Day, Line, and Bound
## Input the data frame, column, and a list of valid entries for that column. 
## For each of these we will also want to drop the null values, so let's build that in so we can consider na entries as invalid. 
## When drop_na = False, this will print the number of null entries, but will not drop null values. 
## When drop = True, the invalid entries are dropped, but the null entries are only dropped if drop_na = True.   
def value_check(df, col, good_entries, drop = False, drop_na = False):
    s = df[col]
    init_row_count = len(df)
    
    # mask is true if entry is invalid (and not null)
    bad_mask = s.notna() & ~s.isin(good_entries)
    
    # count the number of invalid entries
    bad_entry_count = bad_mask.sum()
    
    # count the number of unique invalid entries
    bad_value_count = len(s.loc[bad_mask].unique())
    
    if drop == False:
        print("Number of invalid entries in ", col," : ", bad_entry_count)
        print("Number of (unique) invalid values in ", col," : ", bad_value_count)
    #if drop, then drop the bad entries 
    if drop:
        df = df.loc[~bad_mask]
    # if drop and drop_na, then drop the bad entries and also the na entries
        if drop_na:
            df = df.loc[~s.isna()]
        print(init_row_count - len(df), " rows have been dropped for", col)

    return df

In [184]:
c = ["Time", "Date", "Day"]
for col in c:
    df = null_count(df, col, drop = True)


6  rows have been dropped.
0  rows have been dropped.
0  rows have been dropped.


In [185]:
# This is exactly the six entries we expected so we will proceed with the drop and convert date and time into a single date time object

df["Date"] = pd.to_datetime(
    df["Date"] ,
    errors="coerce"
)

# Convert to a string
df["Time"] = df["Time"].astype("string").str.strip()
# Some time stamps include seconds and some don't but the ones that do are all :00 seconds so lets strip that away 
df["Time"] = df["Time"].str.slice(0, 5)

valid_days = {
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
}
## Check that the days of the week make sense
print("Day entries coincide with the Date entries: ", (df["Day"] == df["Date"].dt.day_name()).all())
df = value_check(df, "Day", valid_days, drop = True, drop_na=True)

# Merge date and time into a single datetime object then drop day and time columns.
df["Datetime"] = pd.to_datetime(
    df["Date"].astype("string") + " " + df["Time"],
    errors="coerce"
)
df["NumDay"] = df["Datetime"].dt.weekday

df = df.drop(columns=c)

Day entries coincide with the Date entries:  True
0  rows have been dropped for Day


Next, Lets clean up the Line and Vehicle columns. These are both numbers but we would like to treat them as strings since they are giving categorical info rather than numerical info.

In [186]:
# Let's standardize the text for any string entries. We will use the same conventions for any strings so let's write a function first.
def clean_str_col(df,col):
    s = df[col]
    clean = (s.astype("string").str.strip().str.upper()
    )
    return clean.astype("string")

    
def num_to_str(df,col):
    s = df[col]
    numer = pd.to_numeric(s, errors="coerce")
    df[col] = s.where(
        numer.isna(),
        numer.astype("Int64").astype("string")
        )
    return clean_str_col(df,col)
    

In [187]:
## Clean up Vehicle and turn into string, then drop any null values
df["Vehicle"] = num_to_str(df,"Vehicle")

df = null_count(df, "Vehicle", drop = True)


4685  rows have been dropped.


In [188]:
## Next, we look at the Line entries, clean them up and turn them into strings and then check for valid entries 
# ## Checking TTC current routes it looks like we have 3 different sets of routes, I will make a dictionary for each to help index these
## Might be helpful for later to have the names and/or the type of route.
df["Line"] = num_to_str(df,"Line")

regular_routes = {
    "501": "Queen",
    "503": "Kingston Rd",
    "504": "King",
    "505": "Dundas",
    "506": "Carlton",
    "507": "Long Branch",
    "508": "Lake Shore",
    "509": "Harbourfront",
    "510": "Spadina",
    "511": "Bathurst",
    "512": "St Clair"
}

##The night routes which correspond to the daytime routes with different first digit
blue_night_routes = {
    "301": "Queen",    
    "304": "King",
    "305": "Dundas",
    "306": "Carlton",
    "310": "Spadina",
    "312": "St Clair"
}


##Don't really understand what these are, but it looks like they might be add ons during rush hour times or something like this? 
limited_service_routes = {
    "507": "Long Branch",
    "508": "Lake Shore",
    "519": "Limited Route"
}

## One dictionary that stores all of them together
route_categories = {
    "regular": regular_routes,
    "blue_night": blue_night_routes,
    "limited_service": limited_service_routes
}

valid_routes = set()
for category in route_categories.values():
    valid_routes.update(category.keys())


## I am going to create a new column called Route Number which will extract the first three digit number from the string
df["Route Number"] = df["Line"].astype("string").str.extract(r"(\d{3})")


In [189]:
df = value_check(df, "Route Number", valid_routes, drop = True, drop_na=True)
df = df.drop(columns="Line")

5101  rows have been dropped for Route Number


In [190]:
## Next we want to clean up station and incident

## One way to standardize the station would be to encode each entry by the "stations id" but I will have to come back for that because idk what these are yet.  
## I will leave it like this for the moment.
## Similarly, we should probably standardize incident using the codes introduced in 2025, but alas I am not sure if we will even use this as a feature so for now I will just do minimal cleaning
df["Station"] = clean_str_col(df,"Station")
df = null_count(df,"Station", drop = True)
df["Incident"] = clean_str_col(df,"Incident")
df = null_count(df,"Incident", drop = True)

209  rows have been dropped.
1  rows have been dropped.


The bound entries are all over the place, I will try to clean up by checking for all of the reasonable variations that occur.

In [191]:
df["Bound"] = clean_str_col(df, "Bound").str.replace("/","").str.replace(" ","")

valid_bounds = {"N", "S", "E", "W", "B", "NB", "SB", "EB", "WB"}

# Lets choose a standard naming convention
# We can probably drop the B portion, but lets just leave it for now
bound_map = {
    "NORTH": "N", "SOUTH": "S", "EAST": "E", "WEST": "W",
    "NORTHBOUND": "NB", "SOUTHBOUND": "SB", "EASTBOUND": "EB", "WESTBOUND": "WB",
    "BN": "NB", "BS": "SB", "BE": "EB", "BW": "WB",        
    "B": "B"
}

df["Bound"] = df["Bound"].replace(bound_map)

df = value_check(df, "Bound", valid_bounds, drop = True, drop_na=True)

13028  rows have been dropped for Bound


We decided that Min Gap and Min Delay are giving very similar information, so let's drop Min Gap and clean up Min Delay. We also make sure that the entries for Min Delay are stored as integers. 

For whatever reason there are a lot of entries where min delay is 0. This is probably due to some kind of rounding, but its not valuable for our purposes so we will also drop all rows where the Min Delay is 0 minutes. 

In [192]:
df = df.drop(columns="Min Gap")
df = null_count(df, "Min Delay", drop = True)
df["Min Delay"] = df["Min Delay"].astype(int)
df = df.loc[(df["Min Delay"]>0)]

53  rows have been dropped.


Lets check for any rows that are duplicates. 

In [193]:
dup_rows = df.duplicated(keep = False)
print('Duplicate rows:', dup_rows.sum())
display(df[dup_rows].sort_values(by = "Datetime"))

Duplicate rows: 399


,Station,Bound,Vehicle,Incident,Min Delay,Datetime,NumDay,Route Number
267,RONCESVALLES AND QUEEN,EB,4065,HELD BY,4,2014-01-15 06:12:00,2,512
268,RONCESVALLES AND QUEEN,EB,4065,HELD BY,4,2014-01-15 06:12:00,2,512
269,QUEEN AND RONCESVALLES,WB,4248,HELD BY,10,2014-01-15 06:15:00,2,501
270,QUEEN AND RONCESVALLES,WB,4248,HELD BY,10,2014-01-15 06:15:00,2,501
755,CARLTON AND SHERBOURNE,WB,4047,HELD BY,14,2014-03-08 08:00:00,5,506
...,...,...,...,...,...,...,...,...
159392,GERRARD AND JONES,E,4472,MTSAN,10,2025-11-02 14:09:00,6,506
159755,KING AND DUFFERIN,E,4442,MTSAN,8,2025-11-12 07:27:00,2,504
159756,KING AND DUFFERIN,E,4442,MTSAN,8,2025-11-12 07:27:00,2,504
161370,ST CLAIR WEST STATION,W,4649,TTO,9,2025-12-24 12:09:00,2,512


There doesn't seem to be any discernable pattern in these so lets go ahead and drop the duplicate rows. We will also reorder the columns to group time properties and location properties, and then sort by time/date. Then we can see how things are looking. 

In [194]:
df = df.drop_duplicates()

df = df[
    ['Datetime', 'NumDay', 'Route Number','Bound', 'Station','Vehicle', 'Incident', 'Min Delay']
]
df = df.sort_values(by=['Datetime'])
print('Rows:', len(df), 'Columns:', df.shape[1])
mem = df.memory_usage(deep=True).sum() / (1024**2)
print(f'Memory MB: {mem:.2f}')
display(df.head(10))
display(df.tail(10))
display(df.sample(10))
display(df.describe(include='all').T)

Rows: 131522 Columns: 8
Memory MB: 13.30


,Datetime,NumDay,Route Number,Bound,Station,Vehicle,Incident,Min Delay
0,2014-01-02 06:31:00,3,505,EB,DUNDAS AND RONCESVALLES,4018,LATE LEAVING GARAGE,4
1,2014-01-02 12:43:00,3,504,EB,KING AND SHAW,4128,UTILIZED OFF ROUTE,20
2,2014-01-02 14:01:00,3,501,WB,KINGSTON ROAD AND BINGHAM,4016,HELD BY,13
3,2014-01-02 14:22:00,3,504,WB,KING ST. AND RONCESVALLES AVE.,4175,INVESTIGATION,7
4,2014-01-02 16:42:00,3,504,EB,KING AND BATHURST,4080,UTILIZED OFF ROUTE,3
5,2014-01-02 17:39:00,3,501,WB,QUEEN AND BEACONSFEILD,4202,HELD BY,7
6,2014-01-02 18:38:00,3,504,EB,RONCESVALLES AND KING STREET WEST,4100,UTILIZED OFF ROUTE,4
7,2014-01-02 19:27:00,3,510,SB,SPADINA AND ST. ANDREWS,4123,INVESTIGATION,20
8,2014-01-03 01:00:00,4,504,WB,BROADVIEW AND QUEEN,4079,UTILIZED OFF ROUTE,7
9,2014-01-03 05:09:00,4,512,EB,BATHURST AND ST. CLAIR,4160,MECHANICAL,3


,Datetime,NumDay,Route Number,Bound,Station,Vehicle,Incident,Min Delay
161630,2025-12-31 20:01:00,2,501,E,QUEEN AND DOWLING,4591,MTAFR,10
161631,2025-12-31 20:16:00,2,507,E,LAKESHORE AND FOURTH,4554,MTAFR,13
161634,2025-12-31 20:57:00,2,506,E,HIGH PARK LOOP,4477,MTSAN,10
161635,2025-12-31 21:02:00,2,504,W,BERKELY AND KING,8645,EFO,9
161637,2025-12-31 21:40:00,2,504,N,BERKELEY AND KING (WE,8787,MFUIR,9
161639,2025-12-31 21:49:00,2,504,W,CHERRY AND FRONT,4643,MTSAN,10
161640,2025-12-31 22:33:00,2,505,N,PARLIAMENT AND DUNDAS,4511,MTTO,10
161641,2025-12-31 22:51:00,2,505,N,BAY AND DUNDAS,4453,STDP,10
161642,2025-12-31 23:31:00,2,501,E,YORK AND ADELAIDE,4547,MTSAN,20
161643,2025-12-31 23:52:00,2,511,W,FLEET AND BATHURST,4481,TTSW,5


,Datetime,NumDay,Route Number,Bound,Station,Vehicle,Incident,Min Delay
11065,2015-01-02 23:19:00,4,504,WB,KING AND JARVIS,4107,MECHANICAL,19
74896,2019-08-20 13:49:00,1,509,WB,QUEENS QUAY AND BATHURST,4488,MECHANICAL,1
133252,2024-01-05 16:59:00,4,501,W,QUEEN AND LOGAN,4437,DIVERSION,10
59063,2018-06-21 17:59:00,3,504,WB,RONCY,4143,GENERAL DELAY,4
114998,2022-10-01 04:29:00,5,510,N,SPADINA STATION,4579,CLEANING - UNSANITARY,10
81339,2020-04-28 22:15:00,1,504,EB,DUNDAS WEST STATION,4599,MECHANICAL,10
51914,2018-01-14 10:00:00,6,505,EB,BROADVIEW/DUNDAS,4053,MECHANICAL,8
43083,2017-05-31 14:30:00,2,501,EB,DUNN AND QUEEN,1665,MECHANICAL,7
114019,2022-09-07 15:51:00,2,511,S,BATHURST STATION,4419,OPERATIONS,9
77884,2019-12-05 08:24:00,3,501,WB,QUEEN/ELMER,4557,HELD BY,18


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
Datetime,131522,NaN,NaN,NaN,2019-12-08 04:50:15.716002,2014-01-02 06:31:00,2017-01-13 07:00:15,2019-09-12 05:57:00,2022-10-21 00:05:30,2025-12-31 23:52:00,NaN
NumDay,131522.0,NaN,NaN,NaN,2.91184,0.0,1.0,3.0,5.0,6.0,1.944573
Route Number,131522,18,501,34027,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Bound,131522,9,WB,32585,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Station,131522,17525,DUNDAS WEST STATION,2438,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Vehicle,131522,3535,0,2167,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Incident,131522,120,MECHANICAL,41667,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Min Delay,131522.0,NaN,NaN,NaN,14.636631,1.0,5.0,8.0,12.0,1400.0,32.388008


In [195]:
df.info()

<class 'pandas.DataFrame'>
Index: 131522 entries, 0 to 161643
Data columns (total 8 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   Datetime      131522 non-null  datetime64[us]
 1   NumDay        131522 non-null  int32         
 2   Route Number  131522 non-null  string        
 3   Bound         131522 non-null  string        
 4   Station       131522 non-null  string        
 5   Vehicle       131522 non-null  string        
 6   Incident      131522 non-null  string        
 7   Min Delay     131522 non-null  int64         
dtypes: datetime64[us](1), int32(1), int64(1), string(5)
memory usage: 13.3 MB


This looks much better. For now, I will store this as a new csv file with the cleaned data that can be used for some EDA. 

In [196]:
df.to_csv('../data/cleaned/clean_2014_2025_data.csv', index=False)